# Monthly PM₂.₅ Compliance Report

**Goal:** Produce a monthly air quality summary suitable for a Local Authority
Annual Status Report, checking PM₂.₅ against WHO 2021 guidelines and UK limit values.

**API keys required:** None (AURN and AQE are freely accessible)

**Aeolus features demonstrated:**
- `find_sites()` for nearest-monitor discovery
- `download()` for data retrieval
- `metrics.aq_stats()` for regulatory statistics
- `metrics.aqi_summary()` with UK DAQI index
- `metrics.aqi_check_who()` for WHO guideline compliance
- `viz.plot_calendar()` for calendar heatmap
- `viz.plot_timeseries()` with guideline overlay

In [ ]:
import aeolus
from aeolus import metrics, viz
from datetime import datetime

import pandas as pd
import matplotlib.pyplot as plt

## 1. Find the Nearest Monitor

A local authority officer needs data for their borough. We use `find_sites()`
to find the closest AURN or AQE monitor to a given location.

In [ ]:
# Find nearest monitors to a borough centre (e.g., Camden Town Hall)
# Search across free UK regulatory networks
sites = aeolus.find_sites(
    ["AURN", "AQE"],
    near=(51.5392, -0.1426),  # Camden
    radius_km=10,
)

print(f"Found {len(sites)} monitors within 10km")
sites[["site_code", "site_name", "source_network", "distance_km"]].head()

In [ ]:
# Select the nearest site (already sorted by distance)
target_site = sites.iloc[0]["site_code"]
target_network = sites.iloc[0]["source_network"]
target_name = sites.iloc[0]["site_name"]

print(f"Using: {target_name} ({target_site}) from {target_network}")
print(f"Distance: {sites.iloc[0]['distance_km']:.1f} km")

## 2. Download Data

In [ ]:
# Download the current year's data
data = aeolus.download(
    target_network,
    sites=[target_site],
    start_date=datetime(2024, 1, 1),
    end_date=datetime(2024, 12, 31),
)

# Filter to PM2.5
pm25 = data[data["measurand"] == "PM2.5"]
print(f"PM\u2082.\u2085 records: {len(pm25):,}")
print(f"Date range: {pm25['date_time'].min()} to {pm25['date_time'].max()}")

## 3. Data Capture Assessment

Data capture rate is critical for regulatory reporting. DEFRA requires \u226575%
for annual statistics to be considered valid.

In [ ]:
# aq_stats automatically calculates data capture
annual = metrics.aq_stats(pm25, pollutant="PM2.5")

capture = annual["data_capture"].iloc[0]
print(f"Annual data capture: {capture:.1%}")
print(f"Status: {'\u2705 Valid' if capture >= 0.75 else '\u26a0\ufe0f Below threshold'}")

annual[["site_code", "year", "data_capture", "annual_mean",
        "max_daily_mean", "p95"]]

## 4. WHO Guideline Compliance

The WHO 2021 guidelines set a PM\u2082.\u2085 annual guideline of 5 \u00b5g/m\u00b3, with
interim targets (IT-1 through IT-4) for countries making progress. The UK
limit value is currently 20 \u00b5g/m\u00b3 (annual mean).

In [ ]:
# Check all WHO targets
for target in ["AQG", "IT-4", "IT-3", "IT-2", "IT-1"]:
    result = metrics.aqi_check_who(pm25, target=target)
    pm_row = result[result["pollutant"] == "PM2.5"]
    if not pm_row.empty:
        row = pm_row.iloc[0]
        status = "\u2705" if row["meets_guideline"] else "\u274c"
        print(f"{status} {target:>4}: {row['guideline_value']:.0f} \u00b5g/m\u00b3 "
              f"(measured: {row['mean_concentration']:.1f} \u00b5g/m\u00b3)")

## 5. UK DAQI Distribution

Calculate daily AQI values and show the distribution across DAQI bands.
This tells the public how many "good" vs "poor" air quality days they experienced.

In [ ]:
# Daily AQI summary
daily_aqi = metrics.aqi_summary(pm25, index="UK_DAQI", freq="D")

# Count days in each category
category_counts = daily_aqi["aqi_category"].value_counts().sort_index()
print("Days in each DAQI band:")
for category, count in category_counts.items():
    print(f"  {category}: {count} days")

## 6. Time Series with Guideline

Plot the full year's data with a WHO guideline overlay to visually
identify periods of poor air quality.

In [ ]:
# Time series with WHO annual guideline
fig = viz.plot_timeseries(
    pm25,
    pollutants=["PM2.5"],
    guideline=5.0,
    guideline_label="WHO AQG (5 \u00b5g/m\u00b3)",
    title=f"PM\u2082.\u2085 at {target_name} ({target_site}) \u2014 2024",
)
plt.show()

## 7. Calendar Heatmap

A calendar heatmap provides an intuitive view of daily pollution levels
across the year. Missing data appears as blank cells.

In [ ]:
# Calendar heatmap of daily mean PM2.5
# First compute daily means using time_average
daily = metrics.time_average(pm25, freq="D")

fig = viz.plot_calendar(
    daily,
    pollutant="PM2.5",
    year=2024,
    title=f"Daily PM\u2082.\u2085 \u2014 {target_name} (2024)",
)
plt.show()

## 8. Monthly Summary Table

Produce a table suitable for inclusion in a Local Authority Annual Status Report.

In [ ]:
# Monthly statistics using time_average for means and data capture
monthly = metrics.time_average(pm25, freq="ME")
monthly_pm = monthly[monthly["measurand"] == "PM2.5"].copy()
monthly_pm["month"] = monthly_pm["date_time"].dt.strftime("%B")

summary = monthly_pm[["month", "value", "data_capture"]].rename(
    columns={"value": "mean_pm25_ugm3", "data_capture": "data_capture_%"}
)
summary["data_capture_%"] = (summary["data_capture_%"] * 100).round(1)
summary["mean_pm25_ugm3"] = summary["mean_pm25_ugm3"].round(1)

print(f"Monthly PM\u2082.\u2085 Summary \u2014 {target_name} ({target_site})")
print("=" * 50)
summary

## Summary

This notebook demonstrated a complete compliance reporting workflow:

1. **Site discovery** with nearest-monitor search
2. **Data capture assessment** using `aq_stats()`
3. **WHO guideline checking** across all interim targets
4. **DAQI band distribution** for public health messaging
5. **Calendar heatmap** for visual communication
6. **Monthly summary table** for regulatory reports

### For a full Annual Status Report
- Download 5 years for trend analysis (see `metrics.trend()`)
- Include NO\u2082 and PM\u2081\u2080 alongside PM\u2082.\u2085
- Compare multiple sites across the borough